# 백오프·큐·동시성 제한

통제된 로컬 HTTP 서버에서 429를 재현하고 재시도 대기와 동시성 제한을 확인합니다.

## 실행 환경 준비

In [1]:
import asyncio, json, random, threading, time
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.error import HTTPError
from urllib.request import urlopen
print("표준 라이브러리 준비 완료")

표준 라이브러리 준비 완료


## 로컬 429 서버

처음 두 요청에는 429를 반환하고 세 번째 요청부터 200을 반환합니다. 외부 제공자에는 요청하지 않습니다.

In [2]:
class Handler(BaseHTTPRequestHandler):
    count = 0
    def do_GET(self):
        Handler.count += 1
        status = 429 if Handler.count <= 2 else 200
        self.send_response(status); self.end_headers(); self.wfile.write(str(status).encode())
    def log_message(self, *args): pass

server = HTTPServer(("127.0.0.1", 0), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
url = f"http://127.0.0.1:{server.server_port}/"
print("로컬 서버 준비")

로컬 서버 준비


## 큐와 동시성 제한

세마포어가 동시에 처리하는 작업 수를 2개로 제한합니다.

In [3]:
active = peak = 0
queue, semaphore, completed, waits = asyncio.Queue(), asyncio.Semaphore(2), [], []
for number in range(6): queue.put_nowait((number, time.perf_counter()))
initial_size = queue.qsize()

async def worker():
    global active, peak
    while not queue.empty():
        number, queued_at = await queue.get()
        async with semaphore:
            waits.append(time.perf_counter() - queued_at)
            active += 1; peak = max(peak, active)
            await asyncio.sleep(0.03); completed.append(number); active -= 1
        queue.task_done()

await asyncio.gather(worker(), worker(), worker())
print("초기 큐:", initial_size, "완료:", len(completed), "최대 동시 실행:", peak)
print("대기 발생 작업:", sum(wait > 0 for wait in waits))
assert queue.empty() and peak <= 2

초기 큐: 6 완료: 6 최대 동시 실행: 2
대기 발생 작업: 6


## 실패 재현

재시도 없이 한 번만 요청하면 429가 그대로 호출자에게 전달됩니다.

In [4]:
try:
    urlopen(url)
except HTTPError as error:
    print("첫 요청 상태:", error.code)
    assert error.code == 429

첫 요청 상태: 429


## 지수 백오프와 지터 적용

시도 횟수가 늘수록 대기 시간을 늘리고 작은 무작위 지연을 더합니다.

In [5]:
def fetch_with_backoff(target, attempts=4):
    waits = []
    for attempt in range(attempts):
        try:
            return urlopen(target).status, waits
        except HTTPError as error:
            if error.code != 429 or attempt == attempts - 1: raise
            delay = 0.05 * (2 ** attempt) + random.uniform(0, 0.01)
            waits.append(round(delay, 3)); time.sleep(delay)

status, waits = fetch_with_backoff(url)
print("최종 상태:", status, "대기:", waits)
server.shutdown(); assert status == 200

최종 상태: 200 대기: [0.052]
